# Chấm submission trên máy local

Mở notebook bằng VS Code/Jupyter và chọn kernel `.venv` Python 3.13.
Cài `requirements.txt` và `ipykernel` vào `.venv` trước khi chạy.
Chỉnh 5 giá trị trong cell cấu hình rồi Run All. Artifact được đọc trực tiếp
trong project, kết quả lưu vào `results/<TRACK>/<ID>/<CONFIG_NAME>/<MODE>/<run_id>/`.
Dev có timeout 300 giây; full chạy theo mặc định chính thức. Không cần upload/giải nén results.


In [ ]:
from pathlib import Path
import sys, importlib.metadata

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src/common/local_grade.py').is_file() and (p/'requirements.txt').is_file()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ thư mục project hoặc notebooks/')
if sys.version_info[:2] != (3, 13) or Path(sys.prefix).resolve() != (ROOT/'.venv').resolve():
    raise RuntimeError('Chọn kernel Python 3.13 của .venv trong project')
EXPECTED_PACKAGES = {'kickmix': '1.0.0', 'numpy': '2.5.4', 'scipy': '1.18.1'}
for package, version in EXPECTED_PACKAGES.items():
    if importlib.metadata.version(package) != version:
        raise RuntimeError(f'Cài lại requirements.txt bằng .venv: {package} cần {version}')
print('Project:', ROOT)
print('Python:', sys.executable)


In [ ]:
TRACK = 'invsqrt'       # invsqrt / dqi / dfthcsos
MODE = 'dev_5min'       # smoke / dev_5min / full
EXPERIMENT_ID = 'INV-BASELINE'
OWNER = 'team-member'
CONFIG_NAME = 'baseline'


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('local_grade', ROOT/'src/common/local_grade.py')
grader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(grader)
OUT = grader.grade(ROOT, TRACK, MODE, EXPERIMENT_ID, OWNER, CONFIG_NAME)


Kết quả đã ở đúng thư mục local, gồm config chấm, metadata, stdout/stderr và
verifier.json khi có output JSON. Provenance được nối từ build_only nếu tìm được
đúng một record khớp hash; nếu thiếu/mơ hồ, agent cần xử lý review.json trước PR.
Báo agent đường dẫn Saved để phân tích và cập nhật tracking. BUILT/dev PASS không
thay cho full PASS. Đổi MODE/CONFIG_NAME rồi chạy lại sẽ tạo run mới.
